In [1]:
# Only use this code block if you are using Google Colab.
# If you are using Jupyter Notebook, please ignore this code block. You can directly upload the file to your Jupyter Notebook file systems.
from google.colab import files

## It will prompt you to select a local file. Click on “Choose Files” then select and upload the file.
## Wait for the file to be 100% uploaded. You should see the name of the file once Colab has uploaded it.
uploaded = files.upload()

Saving things_to_do_all_with_local_with_labels.csv to things_to_do_all_with_local_with_labels.csv
Saving Caro_Album_topics.csv to Caro_Album_topics.csv
Saving Cass_Album_topics.csv to Cass_Album_topics.csv
Saving Melissa_Album_topics.csv to Melissa_Album_topics.csv
Saving Mike_Album_topics.csv to Mike_Album_topics.csv
Saving Paolo_Album_topics.csv to Paolo_Album_topics.csv


# Test

In [ ]:
 """
travel_recommender.py

Requirements (install first):
pip install pandas numpy scikit-learn sentence-transformers

If you don't want embeddings, the script will auto-fallback to TF-IDF (scikit-learn only).
"""

import os
import re
import math
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Try to import sentence-transformers; if not available, we'll use TF-IDF fallback
USE_EMBEDDINGS = True
try:
    from sentence_transformers import SentenceTransformer
except Exception as e:
    print("sentence-transformers not available; falling back to TF-IDF. "
          "Install sentence-transformers for better results: pip install sentence-transformers")
    USE_EMBEDDINGS = False

# -------------------------
# Helper functions
# -------------------------
def normalize_text(s):
    """Lowercase, replace underscores with spaces, remove punctuation (keeps alphanumerics and spaces)."""
    if not isinstance(s, str):
        return ""
    s = s.replace("_", " ")
    s = s.lower()
    s = re.sub(r'[^a-z0-9\s]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def build_person_profile(top_words_series, weights_series=None):
    """
    Combine multiple topic top-word strings into a single profile text for a person.
    - top_words_series: iterable of strings like "landmark, tourist, beach, ..."
    - weights_series: optional iterable of floats (same length) to weight topics
    If weights provided, we repeat topic tokens proportionally to the weight (simple heuristic).
    """
    tokens = []
    if weights_series is None:
        weights_series = [1.0] * len(top_words_series)
    for tw, w in zip(top_words_series, weights_series):
        if not isinstance(tw, str) or not tw.strip():
            continue
        # split on commas, normalize each token
        parts = [normalize_text(p) for p in tw.split(',') if p.strip()]
        # repeat token groups proportional to weight (rounded)
        rep = max(1, int(round(float(w) * 3)))  # scale factor 3 is heuristic
        for _ in range(rep):
            tokens.extend(parts)
    return " ".join(tokens)

def build_article_text(row):
    """
    Compose article text from available fields: title, category, meta_description/body (if present).
    Also include hero_image_url filename tokens as lightweight cues.
    """
    parts = []
    if 'title' in row and isinstance(row['title'], str):
        parts.append(row['title'])
    if 'category' in row and isinstance(row['category'], str):
        parts.append(row['category'])
    # if the full body exists prefer it
    if 'body' in row and isinstance(row['body'], str) and row['body'].strip():
        parts.append(row['body'][:1000])  # limit length
    elif 'meta_description' in row and isinstance(row['meta_description'], str):
        parts.append(row['meta_description'])
    # add hero_image filename tokens as small cue
    if 'hero_image_url' in row and isinstance(row['hero_image_url'], str):
        fname = row['hero_image_url'].split("/")[-1].split(".")[0]
        # split on common delimiters
        tokens = re.split(r'[_\-\./]+', fname)
        tokens = [t for t in tokens if len(t) > 2]
        if tokens:
            parts.append(" ".join(tokens))
    return normalize_text(" ".join(parts))

# -------------------------
# Core recommender
# -------------------------
class TravelRecommender:
    def __init__(self, articles_df, topics_df, embedding_model_name='all-MiniLM-L6-v2'):
        """
        articles_df: DataFrame with article rows (title, category, hero_image_url, detail_url, body/meta_description optional)
        topics_df: DataFrame with columns person, topic_number, top_words, optional topic_weight
        """
        self.articles = articles_df.copy().reset_index(drop=True)
        self.topics = topics_df.copy().reset_index(drop=True)
        # Precompute article_text
        self.articles['article_text'] = self.articles.apply(build_article_text, axis=1)
        self.article_texts = self.articles['article_text'].tolist()
        self.article_ids = self.articles.index.tolist()

        # Setup embedding or TF-IDF
        self.use_embeddings = USE_EMBEDDINGS
        if self.use_embeddings:
            try:
                print("Loading embedding model:", embedding_model_name)
                self.embedder = SentenceTransformer(embedding_model_name)
            except Exception as e:
                print("Failed to load embedding model, falling back to TF-IDF. Error:", e)
                self.use_embeddings = False

        if not self.use_embeddings:
            # TF-IDF vectorizer
            self.vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1,2))
            # Fit on article texts only (we'll transform profiles later into the same space)
            self.article_tfidf = self.vectorizer.fit_transform(self.article_texts)

        else:
            # Precompute article embeddings
            self.article_embeddings = self.embedder.encode(self.article_texts, show_progress_bar=False, convert_to_numpy=True)

    def recommend_for_person(self, person_name, top_n=3, verbose=False):
        # Get topics rows for this person
        p_rows = self.topics[self.topics['person'] == person_name]
        if p_rows.empty:
            raise ValueError(f"No topics found for person '{person_name}'")

        # Build profile text (use topic_weight if present)
        if 'topic_weight' in p_rows.columns:
            weights = p_rows['topic_weight'].tolist()
        else:
            weights = None
        profile_text = build_person_profile(p_rows['top_words'].tolist(), weights)

        # Prepare profile vector or embedding
        if self.use_embeddings:
            profile_vec = self.embedder.encode([profile_text], show_progress_bar=False, convert_to_numpy=True)[0]
            sims = cosine_similarity(profile_vec.reshape(1, -1), self.article_embeddings)[0]
        else:
            profile_tfidf = self.vectorizer.transform([profile_text])
            sims = cosine_similarity(profile_tfidf, self.article_tfidf)[0]

        # Also compute simple keyword overlap (interpretability)
        profile_tokens = set(normalize_text(profile_text).split())
        def overlap_count(article_text):
            art_tokens = set(article_text.split())
            return len(profile_tokens & art_tokens)
        overlaps = [overlap_count(t) for t in self.article_texts]

        # Build results DataFrame
        res_df = self.articles.copy()
        res_df['similarity'] = sims
        res_df['overlap_count'] = overlaps

        # Sort by similarity, then overlap_count
        res_df = res_df.sort_values(['similarity', 'overlap_count'], ascending=[False, False]).reset_index(drop=True)

        # Optionally display verbose info
        if verbose:
            print(f"Profile text for {person_name}:")
            print(profile_text)
            print("\nTop matches (similarity, overlap_count):")
            print(res_df[['title','similarity','overlap_count']].head(top_n))

        # Return top_n with key fields
        top = res_df.head(top_n)[['title','category','detail_url','similarity','overlap_count']]
        return top

# -------------------------
# Example usage: with small sample dataset (from your message)
# -------------------------
if __name__ == "__main__":
    # Example articles: you can replace this with pd.read_csv("articles.csv")
    articles = [
        {
            "title": "14 Things To Do And See In Downtown Houston",
            "category": "SEE AND DO",
            "listing_image_url": "https://cdn-v2.theculturetrip.com/10x/wp-content/uploads/2024/07/shutterstock_2456508113-1-750x500.webp?quality=1",
            "hero_image_url": "https://cdn-v2.theculturetrip.com/1200x630/wp-content/uploads/2024/07/shutterstock_2456508113-1.webp",
            "detail_url": "https://theculturetrip.com/articles/10-locations-to-explore-in-downtown-houston",
            "meta_description": "Downtown Houston highlights: museums, parks, and urban walks."  # optional
        },
        {
            "title": "6 Sensational Places to See Fall Foliage in Texas",
            "category": "RECOMMENDATIONS - OUTDOORS",
            "listing_image_url": "https://cdn-v2.theculturetrip.com/10x/wp-content/uploads/2024/09/shutterstock_2233382101-750x375.webp?quality=1",
            "hero_image_url": "https://cdn-v2.theculturetrip.com/1200x630/wp-content/uploads/2024/09/shutterstock_2233382101.webp",
            "detail_url": "https://theculturetrip.com/articles/where-to-see-fall-foliage-in-texas",
            "meta_description": "Best spots to enjoy fall colors across Texas."
        },
        {
            "title": "8 Cool Things to Do in Dallas at Night",
            "category": "SEE AND DO",
            "listing_image_url": "https://cdn-v2.theculturetrip.com/10x/wp-content/uploads/2018/05/dallas_skyline_ssv4vikottr_luqqryg3xlu18q0ablzbh_rgb_72-750x500.webp?quality=1",
            "hero_image_url": "https://cdn-v2.theculturetrip.com/1200x630/wp-content/uploads/2018/05/dallas_skyline_ssv4vikottr_luqqryg3xlu18q0ablzbh_rgb_72.webp",
            "detail_url": "https://theculturetrip.com/articles/8-cool-things-to-do-in-dallas-at-night",
            "meta_description": "Nightlife and evening attractions in Dallas."
        }
    ]
    articles_df = pd.DataFrame(articles)

    # Example LDA topics for Melissa (you can replace with pd.read_csv("topics.csv"))
    # If you have topic weights (proportions), include a topic_weight column.
    topics = [
        {"person": "Melissa", "topic_number": 0, "top_words": "landmark, tourist, beach, friend, sunny_day, cloudy_sky, winter_clothing, colorful_display, travel, greenery", "topic_weight": 0.2},
        {"person": "Melissa", "topic_number": 1, "top_words": "audience, theater, group_photo, modern_architecture, winter_clothing, landmark, greenery, viewpoint, sunset, clear_sky", "topic_weight": 0.15},
        {"person": "Melissa", "topic_number": 2, "top_words": "friend_hangout, colorful_display, winter_clothing, landmark, tourist, travel, greenery, sunset, clear_sky, beach", "topic_weight": 0.25},
        {"person": "Melissa", "topic_number": 3, "top_words": "travel, clear_sky, modern_architecture, beach, colorful_display, greenery, sunset, viewpoint, landmark, sunny_day", "topic_weight": 0.25},
        {"person": "Melissa", "topic_number": 4, "top_words": "nature, viewpoint, sunset, greenery, cloudy_sky, sunny_day, friend, group_photo, travel, colorful_display", "topic_weight": 0.15},
    ]
    topics_df = pd.DataFrame(topics)

    # Instantiate recommender
    rec = TravelRecommender(articles_df, topics_df)

    # Get top 3 for Melissa
    top3 = rec.recommend_for_person("Melissa", top_n=3, verbose=True)
    print("\nTop 3 recommendations (final):")
    print(top3.to_string(index=False))


Loading embedding model: all-MiniLM-L6-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Profile text for Melissa:
landmark tourist beach friend sunny day cloudy sky winter clothing colorful display travel greenery audience theater group photo modern architecture winter clothing landmark greenery viewpoint sunset clear sky friend hangout colorful display winter clothing landmark tourist travel greenery sunset clear sky beach travel clear sky modern architecture beach colorful display greenery sunset viewpoint landmark sunny day nature viewpoint sunset greenery cloudy sky sunny day friend group photo travel colorful display

Top matches (similarity, overlap_count):
                                               title  similarity  \
0             8 Cool Things to Do in Dallas at Night    0.472100   
1        14 Things To Do And See In Downtown Houston    0.387760   
2  6 Sensational Places to See Fall Foliage in Texas    0.367616   

   overlap_count  
0              0  
1              0  
2              0  

Top 3 recommendations (final):
                                   

#

In [ ]:
web_labels_df = pd.read_csv('things_to_do_all_with_local_with_labels.csv')
web_labels_df.head()

,title,category,listing_image_url,hero_image_url,detail_url,local_path,labels_openai
0,14 Things To Do And See In Downtown Houston,SEE AND DO,https://cdn-v2.theculturetrip.com/10x/wp-conte...,https://cdn-v2.theculturetrip.com/1200x630/wp-...,https://theculturetrip.com/articles/10-locatio...,C:\Users\micha\OneDrive\Documents\Fall Semeste...,"cityscape, skyline, skyscrapers, highway, ligh..."
1,6 Sensational Places to See Fall Foliage in Texas,RECOMMENDATIONS - OUTDOORS,https://cdn-v2.theculturetrip.com/10x/wp-conte...,https://cdn-v2.theculturetrip.com/1200x630/wp-...,https://theculturetrip.com/articles/where-to-s...,C:\Users\micha\OneDrive\Documents\Fall Semeste...,"autumn, fall foliage, orange leaves, yellow le..."
2,8 Cool Things to Do in Dallas at Night,SEE AND DO,https://cdn-v2.theculturetrip.com/10x/wp-conte...,https://cdn-v2.theculturetrip.com/1200x630/wp-...,https://theculturetrip.com/articles/8-cool-thi...,C:\Users\micha\OneDrive\Documents\Fall Semeste...,"cityscape, skyline, skyscrapers, twilight, sun..."
3,"Quiet Escapes In Houston, Texas",SEE AND DO,https://cdn-v2.theculturetrip.com/10x/wp-conte...,https://cdn-v2.theculturetrip.com/1200x630/wp-...,https://theculturetrip.com/articles/10-quiet-e...,C:\Users\micha\OneDrive\Documents\Fall Semeste...,"city skyline, skyscrapers, park, people, grass..."
4,Best Weekend Getaways From Dallas & Fort Worth,RECOMMENDATIONS - ATTRACTIONS,https://cdn-v2.theculturetrip.com/10x/wp-conte...,https://cdn-v2.theculturetrip.com/1200x630/wp-...,https://theculturetrip.com/articles/10-best-we...,C:\Users\micha\OneDrive\Documents\Fall Semeste...,"cityscape, skyline, buildings, tower, sunset, ..."


In [ ]:
melissa_df = pd.read_csv('Melissa_Album_topics.csv')
melissa_df.head()

,person,topic_number,top_words
0,Melissa's Album,0,"landmark, tourist, beach, friend, sunny_day, c..."
1,Melissa's Album,1,"audience, theater, group_photo, modern_archite..."
2,Melissa's Album,2,"friend_hangout, colorful_display, winter_cloth..."
3,Melissa's Album,3,"travel, clear_sky, modern_architecture, beach,..."
4,Melissa's Album,4,"nature, viewpoint, sunset, greenery, cloudy_sk..."


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# --- Recommendation Functions (No change needed here) ---
def recommend_by_labels(web_data, user_topics):
    """
    Recommends top 3 destinations based on cosine similarity of labels.
    """
    all_labels = list(web_data['labels_openai']) + [user_topics]
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(all_labels)
    user_vector = tfidf_matrix[-1]
    web_vectors = tfidf_matrix[:-1]
    similarities = cosine_similarity(user_vector, web_vectors).flatten()
    top_indices = similarities.argsort()[-3:][::-1]
    recommendations = web_data.iloc[top_indices]
    return recommendations[['title', 'detail_url']]

def recommend_by_labels_and_title(web_data, user_topics):
    """
    Recommends top 3 destinations based on cosine similarity of labels and titles.
    """
    web_data['combined_text'] = web_data['title'] + ' ' + web_data['labels_openai']
    all_text = list(web_data['combined_text']) + [user_topics]
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(all_text)
    user_vector = tfidf_matrix[-1]
    web_vectors = tfidf_matrix[:-1]
    similarities = cosine_similarity(user_vector, web_vectors).flatten()
    top_indices = similarities.argsort()[-3:][::-1]
    recommendations = web_data.iloc[top_indices]
    return recommendations[['title', 'detail_url']]


# --- Main execution ---

# Load the datasets
web_df = pd.read_csv('things_to_do_all_with_local_with_labels.csv')
user_df = pd.read_csv('Melissa_Album_topics.csv')

# *** MODIFIED PART ***
# Combine all topic words from the 'top_words' column into a single string
# We use .str.cat() to concatenate the strings with a comma separator
user_topics_str = user_df['top_words'].str.cat(sep=', ')

print("--- Melissa's Combined Topic Profile ---")
print(user_topics_str)
print("-" * 40)


# Get recommendations using the combined topic string
recommendations_labels_only = recommend_by_labels(web_df.copy(), user_topics_str)
recommendations_labels_and_title = recommend_by_labels_and_title(web_df.copy(), user_topics_str)

# Print the results
print("\nRecommendations for Melissa (based on labels only):")
print(recommendations_labels_only)
print("\nRecommendations for Melissa (based on labels and title):")
print(recommendations_labels_and_title)

# --- EXPORT TO CSV (NEW PART) ---
recommendations_labels_only.to_csv('recommendations_labels_only.csv', index=False)
recommendations_labels_and_title.to_csv('recommendations_labels_and_title.csv', index=False)

print("\n✅ Recommendations successfully exported to:")
print("- recommendations_labels_only.csv")
print("- recommendations_labels_and_title.csv")

--- Melissa's Combined Topic Profile ---
landmark, tourist, beach, friend, sunny_day, cloudy_sky, winter_clothing, colorful_display, travel, greenery, audience, theater, group_photo, modern_architecture, winter_clothing, landmark, greenery, viewpoint, sunset, clear_sky, friend_hangout, colorful_display, winter_clothing, landmark, tourist, travel, greenery, sunset, clear_sky, beach, travel, clear_sky, modern_architecture, beach, colorful_display, greenery, sunset, viewpoint, landmark, sunny_day, nature, viewpoint, sunset, greenery, cloudy_sky, sunny_day, friend, group_photo, travel, colorful_display
----------------------------------------

Recommendations for Melissa (based on labels only):
                                                 title  \
86   The 10 Best Things To Do And See In East Downt...   
178              Finding Secret Beach In Austin, Texas   
24   Top Haunted Places in Texas for Would-Be Ghost...   

                                            detail_url  
86   https

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import os

# --- Recommendation Functions (Using Bigrams) ---

def recommend_by_labels(web_data, user_topics):
    """Recommends using bigrams from the 'labels' column."""
    # Ensure all labels are strings, fill missing values with empty string
    web_data['labels_openai'] = web_data['labels_openai'].fillna('')
    all_labels = list(web_data['labels_openai']) + [user_topics]

    # Use ngram_range=(2, 2) to specify bigrams
    vectorizer = TfidfVectorizer(ngram_range=(2, 2))
    tfidf_matrix = vectorizer.fit_transform(all_labels)

    user_vector = tfidf_matrix[-1]
    web_vectors = tfidf_matrix[:-1]
    similarities = cosine_similarity(user_vector, web_vectors).flatten()
    top_indices = similarities.argsort()[-3:][::-1]
    recommendations = web_data.iloc[top_indices]
    return recommendations[['title', 'detail_url']]

def recommend_by_labels_and_title(web_data, user_topics):
    """Recommends using bigrams from the 'title' and 'labels' columns."""
    # Ensure all text data are strings, fill missing values
    web_data['title'] = web_data['title'].fillna('')
    web_data['labels_openai'] = web_data['labels_openai'].fillna('')
    web_data['combined_text'] = web_data['title'] + ' ' + web_data['labels_openai']
    all_text = list(web_data['combined_text']) + [user_topics]

    # Use ngram_range=(2, 2) to specify bigrams
    vectorizer = TfidfVectorizer(ngram_range=(2, 2))
    tfidf_matrix = vectorizer.fit_transform(all_text)

    user_vector = tfidf_matrix[-1]
    web_vectors = tfidf_matrix[:-1]
    similarities = cosine_similarity(user_vector, web_vectors).flatten()
    top_indices = similarities.argsort()[-3:][::-1]
    recommendations = web_data.iloc[top_indices]
    return recommendations[['title', 'detail_url']]


# --- Main execution with a loop ---

# Define the list of individuals to process
individuals = ['Melissa', 'Caro', 'Cass', 'Mike', 'Paolo']

# Load your actual web scraped data
try:
    web_df = pd.read_csv('things_to_do_all_with_local_with_labels.csv')
except FileNotFoundError:
    print("❌ Error: Make sure 'things_to_do_all_with_local_with_labels.csv' is in the correct directory.")
    exit()


# Lists to store the results from each iteration
all_recs_labels = []
all_recs_combined = []

for person_name in individuals:
    # Construct the filename for each person, e.g., "Melissa_Album_topics.csv"
    topic_filename = f'{person_name}_Album_topics.csv'

    if os.path.exists(topic_filename):
        print(f"Processing recommendations for {person_name}...")
        user_df = pd.read_csv(topic_filename)

        # Combine all topic words into a single string for matching
        user_topics_str = user_df['top_words'].str.cat(sep=', ')

        # Get recommendations using both methods
        recs_labels = recommend_by_labels(web_df.copy(), user_topics_str)
        recs_combined = recommend_by_labels_and_title(web_df.copy(), user_topics_str)

        # Add a 'person' column to identify the recommendations
        recs_labels['person'] = person_name
        recs_combined['person'] = person_name

        # Append the results to our master lists
        all_recs_labels.append(recs_labels)
        all_recs_combined.append(recs_combined)
    else:
        print(f"⚠️ Warning: Could not find topic file '{topic_filename}'. Skipping {person_name}.")


# --- Final Consolidation and Export ---

if all_recs_combined:
    # Concatenate all the individual DataFrames into single master DataFrames
    final_recs_labels_df = pd.concat(all_recs_labels, ignore_index=True)
    final_recs_combined_df = pd.concat(all_recs_combined, ignore_index=True)

    # Reorder columns for better readability
    final_recs_labels_df = final_recs_labels_df[['person', 'title', 'detail_url']]
    final_recs_combined_df = final_recs_combined_df[['person', 'title', 'detail_url']]

    # Export the final DataFrames to new CSV files
    final_recs_labels_df.to_csv('all_recommendations_labels_bigrams.csv', index=False)
    final_recs_combined_df.to_csv('all_recommendations_combined_bigrams.csv', index=False)

    print("\n✅ All recommendations using bigrams processed and exported successfully!")
    print(" - all_recommendations_labels_bigrams.csv")
    print(" - all_recommendations_combined_bigrams.csv")
else:
    print("\nNo topic files were found. No recommendations were generated.")

Processing recommendations for Melissa...
Processing recommendations for Caro...
Processing recommendations for Cass...
Processing recommendations for Mike...
Processing recommendations for Paolo...

✅ All recommendations using bigrams processed and exported successfully!
 - all_recommendations_labels_bigrams.csv
 - all_recommendations_combined_bigrams.csv


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import os

# --- Recommendation Functions (Updated for new column names and to return scores) ---

def recommend_by_labels(web_data, user_topics):
    """Recommends using bigrams from the 'labels_openai' column and returns scores."""
    # Use the new 'labels_openai' column
    web_data['labels_openai'] = web_data['labels_openai'].fillna('')
    all_labels = list(web_data['labels_openai']) + [user_topics]

    vectorizer = TfidfVectorizer(ngram_range=(2, 2))
    tfidf_matrix = vectorizer.fit_transform(all_labels)

    user_vector = tfidf_matrix[-1]
    web_vectors = tfidf_matrix[:-1]
    similarities = cosine_similarity(user_vector, web_vectors).flatten()

    top_indices = similarities.argsort()[-3:][::-1]
    top_scores = similarities[top_indices]

    recommendations = web_data.iloc[top_indices].copy()
    recommendations['similarity_score'] = top_scores

    # Return the new 'detail_url' column
    return recommendations[['title', 'detail_url', 'similarity_score']]

def recommend_by_labels_and_title(web_data, user_topics):
    """Recommends using bigrams from 'title' and 'labels_openai' and returns scores."""
    web_data['title'] = web_data['title'].fillna('')
    # Use the new 'labels_openai' column
    web_data['labels_openai'] = web_data['labels_openai'].fillna('')
    web_data['combined_text'] = web_data['title'] + ' ' + web_data['labels_openai']
    all_text = list(web_data['combined_text']) + [user_topics]

    vectorizer = TfidfVectorizer(ngram_range=(2, 2))
    tfidf_matrix = vectorizer.fit_transform(all_text)

    user_vector = tfidf_matrix[-1]
    web_vectors = tfidf_matrix[:-1]
    similarities = cosine_similarity(user_vector, web_vectors).flatten()

    top_indices = similarities.argsort()[-3:][::-1]
    top_scores = similarities[top_indices]

    recommendations = web_data.iloc[top_indices].copy()
    recommendations['similarity_score'] = top_scores

    # Return the new 'detail_url' column
    return recommendations[['title', 'detail_url', 'similarity_score']]


# --- Main execution with a loop ---

individuals = ['Melissa', 'Caro', 'Cass', 'Mike', 'Paolo']

try:
    web_df = pd.read_csv('things_to_do_all_with_local_with_labels.csv')
except FileNotFoundError:
    print("❌ Error: Make sure 'things_to_do_all_with_local_with_labels.csv' is in the correct directory.")
    exit()

all_recs_labels = []
all_recs_combined = []

for person_name in individuals:
    topic_filename = f'{person_name}_Album_topics.csv'

    if os.path.exists(topic_filename):
        print(f"Processing recommendations for {person_name}...")
        user_df = pd.read_csv(topic_filename)
        user_topics_str = user_df['top_words'].str.cat(sep=', ')

        recs_labels = recommend_by_labels(web_df.copy(), user_topics_str)
        recs_combined = recommend_by_labels_and_title(web_df.copy(), user_topics_str)

        recs_labels['person'] = person_name
        recs_combined['person'] = person_name

        all_recs_labels.append(recs_labels)
        all_recs_combined.append(recs_combined)
    else:
        print(f"⚠️ Warning: Could not find topic file '{topic_filename}'. Skipping {person_name}.")


# --- Final Consolidation and Export ---

if all_recs_combined:
    final_recs_labels_df = pd.concat(all_recs_labels, ignore_index=True)
    final_recs_combined_df = pd.concat(all_recs_combined, ignore_index=True)

    # Update the column order for the final CSV
    cols_order = ['person', 'title', 'detail_url', 'similarity_score']
    final_recs_labels_df = final_recs_labels_df[cols_order]
    final_recs_combined_df = final_recs_combined_df[cols_order]

    # Export to new CSV files
    final_recs_labels_df.to_csv('all_recommendations_labels_bigrams_with_scores.csv', index=False)
    final_recs_combined_df.to_csv('all_recommendations_combined_bigrams_with_scores.csv', index=False)

    print("\n✅ All recommendations with scores processed and exported successfully!")
    print(" - all_recommendations_labels_bigrams_with_scores.csv")
    print(" - all_recommendations_combined_bigrams_with_scores.csv")
else:
    print("\nNo topic files were found. No recommendations were generated.")

Processing recommendations for Melissa...
Processing recommendations for Caro...
Processing recommendations for Cass...
Processing recommendations for Mike...
Processing recommendations for Paolo...

✅ All recommendations with scores processed and exported successfully!
 - all_recommendations_labels_bigrams_with_scores.csv
 - all_recommendations_combined_bigrams_with_scores.csv


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import os

# --- Recommendation Functions (Updated to use Unigrams and return scores) ---

def recommend_by_labels(web_data, user_topics):
    """Recommends using unigrams from the 'labels_openai' column and returns scores."""
    # Use the new 'labels_openai' column
    web_data['labels_openai'] = web_data['labels_openai'].fillna('')
    all_labels = list(web_data['labels_openai']) + [user_topics]

    # Vectorizer now uses default unigrams (ngram_range removed)
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(all_labels)

    user_vector = tfidf_matrix[-1]
    web_vectors = tfidf_matrix[:-1]
    similarities = cosine_similarity(user_vector, web_vectors).flatten()

    top_indices = similarities.argsort()[-3:][::-1]
    top_scores = similarities[top_indices]

    recommendations = web_data.iloc[top_indices].copy()
    recommendations['similarity_score'] = top_scores

    # Return the new 'detail_url' column
    return recommendations[['title', 'detail_url', 'similarity_score']]

def recommend_by_labels_and_title(web_data, user_topics):
    """Recommends using unigrams from 'title' and 'labels_openai' and returns scores."""
    web_data['title'] = web_data['title'].fillna('')
    # Use the new 'labels_openai' column
    web_data['labels_openai'] = web_data['labels_openai'].fillna('')
    web_data['combined_text'] = web_data['title'] + ' ' + web_data['labels_openai']
    all_text = list(web_data['combined_text']) + [user_topics]

    # Vectorizer now uses default unigrams (ngram_range removed)
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(all_text)

    user_vector = tfidf_matrix[-1]
    web_vectors = tfidf_matrix[:-1]
    similarities = cosine_similarity(user_vector, web_vectors).flatten()

    top_indices = similarities.argsort()[-3:][::-1]
    top_scores = similarities[top_indices]

    recommendations = web_data.iloc[top_indices].copy()
    recommendations['similarity_score'] = top_scores

    # Return the new 'detail_url' column
    return recommendations[['title', 'detail_url', 'similarity_score']]


# --- Main execution with a loop ---

individuals = ['Melissa', 'Caro', 'Cass', 'Mike', 'Paolo']

try:
    web_df = pd.read_csv('things_to_do_all_with_local_with_labels.csv')
except FileNotFoundError:
    print("❌ Error: Make sure 'things_to_do_all_with_local_with_labels.csv' is in the correct directory.")
    exit()

all_recs_labels = []
all_recs_combined = []

for person_name in individuals:
    topic_filename = f'{person_name}_Album_topics.csv'

    if os.path.exists(topic_filename):
        print(f"Processing recommendations for {person_name}...")
        user_df = pd.read_csv(topic_filename)
        user_topics_str = user_df['top_words'].str.cat(sep=', ')

        recs_labels = recommend_by_labels(web_df.copy(), user_topics_str)
        recs_combined = recommend_by_labels_and_title(web_df.copy(), user_topics_str)

        recs_labels['person'] = person_name
        recs_combined['person'] = person_name

        all_recs_labels.append(recs_labels)
        all_recs_combined.append(recs_combined)
    else:
        print(f"⚠️ Warning: Could not find topic file '{topic_filename}'. Skipping {person_name}.")


# --- Final Consolidation and Export ---

if all_recs_combined:
    final_recs_labels_df = pd.concat(all_recs_labels, ignore_index=True)
    final_recs_combined_df = pd.concat(all_recs_combined, ignore_index=True)

    cols_order = ['person', 'title', 'detail_url', 'similarity_score']
    final_recs_labels_df = final_recs_labels_df[cols_order]
    final_recs_combined_df = final_recs_combined_df[cols_order]

    # Updated filenames to reflect unigram use
    final_recs_labels_df.to_csv('all_recommendations_labels_with_scores.csv', index=False)
    final_recs_combined_df.to_csv('all_recommendations_combined_with_scores.csv', index=False)

    print("\n✅ All recommendations with scores processed and exported successfully!")
    print(" - all_recommendations_labels_with_scores.csv")
    print(" - all_recommendations_combined_with_scores.csv")
else:
    print("\nNo topic files were found. No recommendations were generated.")

Processing recommendations for Melissa...
Processing recommendations for Caro...
Processing recommendations for Cass...
Processing recommendations for Mike...
Processing recommendations for Paolo...

✅ All recommendations with scores processed and exported successfully!
 - all_recommendations_labels_with_scores.csv
 - all_recommendations_combined_with_scores.csv


In [ ]:
!pip install spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 65.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


# Work embeddings

In [ ]:
import pandas as pd
import spacy
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import os

# --- Load the spaCy word embedding model ---
# This model converts text into meaningful vectors.
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    print("❌ Error: SpaCy model 'en_core_web_sm' not found.")
    print("Please run: python -m spacy download en_core_web_sm")
    exit()


# --- Recommendation Functions (Using Word Embeddings) ---

def get_embedding(text):
    """Generates a document vector by averaging word vectors."""
    # The nlp object processes the text and returns a doc object with a vector.
    return nlp(text).vector

def recommend_by_labels(web_data, user_topics_vector):
    """Recommends using word embeddings on the 'labels_openai' column."""
    web_data['labels_openai'] = web_data['labels_openai'].fillna('')

    # Generate vectors for all travel destinations' labels
    # The .tolist() is used for compatibility with cosine_similarity
    web_vectors = np.array(web_data['labels_openai'].apply(get_embedding).tolist())

    # Calculate cosine similarity between the user's vector and all destination vectors
    similarities = cosine_similarity(user_topics_vector.reshape(1, -1), web_vectors).flatten()

    top_indices = similarities.argsort()[-3:][::-1]
    top_scores = similarities[top_indices]

    recommendations = web_data.iloc[top_indices].copy()
    recommendations['similarity_score'] = top_scores

    return recommendations[['title', 'detail_url', 'similarity_score']]

def recommend_by_labels_and_title(web_data, user_topics_vector):
    """Recommends using word embeddings on 'title' and 'labels_openai'."""
    web_data['title'] = web_data['title'].fillna('')
    web_data['labels_openai'] = web_data['labels_openai'].fillna('')
    web_data['combined_text'] = web_data['title'] + ' ' + web_data['labels_openai']

    # Generate vectors for all travel destinations' combined text
    web_vectors = np.array(web_data['combined_text'].apply(get_embedding).tolist())

    # Calculate cosine similarity
    similarities = cosine_similarity(user_topics_vector.reshape(1, -1), web_vectors).flatten()

    top_indices = similarities.argsort()[-3:][::-1]
    top_scores = similarities[top_indices]

    recommendations = web_data.iloc[top_indices].copy()
    recommendations['similarity_score'] = top_scores

    return recommendations[['title', 'detail_url', 'similarity_score']]


# --- Main execution with a loop ---

individuals = ['Melissa', 'Caro', 'Cass', 'Mike', 'Paolo']

try:
    web_df = pd.read_csv('things_to_do_all_with_local_with_labels.csv')
except FileNotFoundError:
    print("❌ Error: Make sure 'things_to_do_all_with_local_with_labels.csv' is in the correct directory.")
    exit()

all_recs_labels = []
all_recs_combined = []

for person_name in individuals:
    topic_filename = f'{person_name}_Album_topics.csv'

    if os.path.exists(topic_filename):
        print(f"Processing recommendations for {person_name}...")
        user_df = pd.read_csv(topic_filename)
        user_topics_str = user_df['top_words'].str.cat(sep=', ')

        # Generate the user's vector once
        user_vector = get_embedding(user_topics_str)

        recs_labels = recommend_by_labels(web_df.copy(), user_vector)
        recs_combined = recommend_by_labels_and_title(web_df.copy(), user_vector)

        recs_labels['person'] = person_name
        recs_combined['person'] = person_name

        all_recs_labels.append(recs_labels)
        all_recs_combined.append(recs_combined)
    else:
        print(f"⚠️ Warning: Could not find topic file '{topic_filename}'. Skipping {person_name}.")


# --- Final Consolidation and Export ---

if all_recs_combined:
    final_recs_labels_df = pd.concat(all_recs_labels, ignore_index=True)
    final_recs_combined_df = pd.concat(all_recs_combined, ignore_index=True)

    cols_order = ['person', 'title', 'detail_url', 'similarity_score']
    final_recs_labels_df = final_recs_labels_df[cols_order]
    final_recs_combined_df = final_recs_combined_df[cols_order]

    # Updated filenames to reflect word embedding use
    final_recs_labels_df.to_csv('all_recommendations_embeddings_labels_scores.csv', index=False)
    final_recs_combined_df.to_csv('all_recommendations_embeddings_combined_scores.csv', index=False)

    print("\n✅ All recommendations using word embeddings processed and exported successfully!")
    print(" - all_recommendations_embeddings_labels_scores.csv")
    print(" - all_recommendations_embeddings_combined_scores.csv")
else:
    print("\nNo topic files were found. No recommendations were generated.")

Processing recommendations for Melissa...
Processing recommendations for Caro...
Processing recommendations for Cass...
Processing recommendations for Mike...
Processing recommendations for Paolo...

✅ All recommendations using word embeddings processed and exported successfully!
 - all_recommendations_embeddings_labels_scores.csv
 - all_recommendations_embeddings_combined_scores.csv


In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import os
import re  # Import regex library for finding words

# --- Helper Function ---
def get_matching_words(user_topics, destination_text):
    """
    Finds and returns the common words between two strings.
    """
    try:
        # Use re.findall(r'\w+', ...) to extract clean words (alphanumeric)
        # We use .lower() to ensure the match is case-insensitive
        user_words = set(re.findall(r'\w+', user_topics.lower()))
        dest_words = set(re.findall(r'\w+', str(destination_text).lower())) # Use str() for safety

        # Find the intersection of the two sets
        matching = user_words.intersection(dest_words)

        # Return as a comma-separated string, sorted alphabetically
        return ", ".join(sorted(list(matching)))
    except Exception as e:
        print(f"Error finding matching words: {e}")
        return "" # Return empty string if anything goes wrong

# --- Updated Recommendation Functions ---

def recommend_by_labels(web_data, user_topics):
    """Recommends using unigrams from the 'labels_openai' column and returns scores."""
    web_data['labels_openai'] = web_data['labels_openai'].fillna('')
    all_labels = list(web_data['labels_openai']) + [user_topics]

    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(all_labels)

    user_vector = tfidf_matrix[-1]
    web_vectors = tfidf_matrix[:-1]
    similarities = cosine_similarity(user_vector, web_vectors).flatten()

    top_indices = similarities.argsort()[-3:][::-1]
    top_scores = similarities[top_indices]

    recommendations = web_data.iloc[top_indices].copy()
    recommendations['similarity_score'] = top_scores

    # --- ADD MATCHING WORDS ---
    recommendations['matching_words'] = recommendations['labels_openai'].apply(
        lambda dest_text: get_matching_words(user_topics, dest_text)
    )

    # Return with the new column
    return recommendations[['title', 'detail_url', 'similarity_score', 'matching_words']]

def recommend_by_labels_and_title(web_data, user_topics):
    """Recommends using unigrams from 'title' and 'labels_openai' and returns scores."""
    web_data['title'] = web_data['title'].fillna('')
    web_data['labels_openai'] = web_data['labels_openai'].fillna('')
    web_data['combined_text'] = web_data['title'] + ' ' + web_data['labels_openai']
    all_text = list(web_data['combined_text']) + [user_topics]

    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(all_text)

    user_vector = tfidf_matrix[-1]
    web_vectors = tfidf_matrix[:-1]
    similarities = cosine_similarity(user_vector, web_vectors).flatten()

    top_indices = similarities.argsort()[-3:][::-1]
    top_scores = similarities[top_indices]

    recommendations = web_data.iloc[top_indices].copy()
    recommendations['similarity_score'] = top_scores

    # --- ADD MATCHING WORDS (using 'combined_text') ---
    recommendations['matching_words'] = recommendations['combined_text'].apply(
        lambda dest_text: get_matching_words(user_topics, dest_text)
    )

    # Return with the new column
    return recommendations[['title', 'detail_url', 'similarity_score', 'matching_words']]


# --- Main execution with a loop ---

individuals = ['Melissa', 'Caro', 'Cass', 'Mike', 'Paolo']

try:
    web_df = pd.read_csv('things_to_do_all_with_local_with_labels.csv')
except FileNotFoundError:
    print("❌ Error: Make sure 'things_to_do_all_with_local_with_labels.csv' is in the correct directory.")
    exit()

all_recs_labels = []
all_recs_combined = []

for person_name in individuals:
    topic_filename = f'{person_name}_Album_topics.csv'

    if os.path.exists(topic_filename):
        print(f"Processing recommendations for {person_name}...")
        user_df = pd.read_csv(topic_filename)
        user_topics_str = user_df['top_words'].str.cat(sep=', ')

        recs_labels = recommend_by_labels(web_df.copy(), user_topics_str)
        recs_combined = recommend_by_labels_and_title(web_df.copy(), user_topics_str)

        recs_labels['person'] = person_name
        recs_combined['person'] = person_name

        all_recs_labels.append(recs_labels)
        all_recs_combined.append(recs_combined)
    else:
        print(f"⚠️ Warning: Could not find topic file '{topic_filename}'. Skipping {person_name}.")


# --- Final Consolidation and Export ---

if all_recs_combined:
    final_recs_labels_df = pd.concat(all_recs_labels, ignore_index=True)
    final_recs_combined_df = pd.concat(all_recs_combined, ignore_index=True)

    # Updated column order to include matching_words
    cols_order = ['person', 'title', 'detail_url', 'similarity_score', 'matching_words']
    final_recs_labels_df = final_recs_labels_df[cols_order]
    final_recs_combined_df = final_recs_combined_df[cols_order]

    # Updated filenames to reflect unigram use
    final_recs_labels_df.to_csv('all_recommendations_labels_with_scores.csv', index=False)
    final_recs_combined_df.to_csv('all_recommendations_combined_with_scores.csv', index=False)

    print("\n✅ All recommendations with scores and matching words processed and exported successfully!")
    print(" - all_recommendations_labels_with_scores.csv")
    print(" - all_recommendations_combined_with_scores.csv")
else:
    print("\nNo topic files were found. No recommendations were generated.")

Processing recommendations for Melissa...
Processing recommendations for Caro...
Processing recommendations for Cass...
Processing recommendations for Mike...
Processing recommendations for Paolo...

✅ All recommendations with scores and matching words processed and exported successfully!
 - all_recommendations_labels_with_scores.csv
 - all_recommendations_combined_with_scores.csv
